# Entity Identification Pipeline for Co-Pilot Agent

## Overview
This notebook implements two approaches for predicting relevant entities from user queries:

1. **Fine-tuned Classifier**: Train DistilBERT on query text, inference on query text only
2. **Fine-tuned Classifier + RAG**: Same trained model, but at inference add retrieved similar examples

## Entity Types
- CDR (Call Detail Records)
- Phone
- Web Activity
- Web Actor
- Person
- Investigation
- Insight
- Report
- EVisa Request

## 1. Setup and Dependencies

In [ ]:
# Install required packages
# !pip install pandas numpy scikit-learn torch transformers sentence-transformers accelerate

import pandas as pd
import numpy as np
import json
import ast
from typing import List, Dict, Tuple
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, hamming_loss, jaccard_score
)

# Deep learning imports
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from sentence_transformers import SentenceTransformer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 2. Load and Preprocess Data

In [ ]:
# Load datasets
user_queries_df = pd.read_csv('user_queries.csv')
fields_description_df = pd.read_csv('fields_description.csv')

print(f"User Queries: {len(user_queries_df)} rows")
print(f"Fields Description: {len(fields_description_df)} rows")
print(f"\nEntity types in fields_description: {fields_description_df['entity_name'].unique().tolist()}")

In [ ]:
def safe_parse_json(json_str: str) -> dict:
    """Parse JSON string, handling Python dict format."""
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(json_str)
        except (ValueError, SyntaxError):
            return {}

def extract_entities(json_obj: dict) -> List[str]:
    """Extract entities from entityType and relationTargetType keys."""
    entities = set()
    
    if 'entityType' in json_obj:
        entities.add(json_obj['entityType'])
    
    def search_statements(statements):
        if not statements:
            return
        for stmt in statements:
            if isinstance(stmt, dict):
                params = stmt.get('parameters', {})
                if 'relationTargetType' in params:
                    targets = params['relationTargetType']
                    if isinstance(targets, list):
                        entities.update(targets)
                    else:
                        entities.add(targets)
                if 'statements' in stmt:
                    search_statements(stmt['statements'])
    
    if 'statements' in json_obj:
        search_statements(json_obj['statements'])
    
    return sorted(list(entities))

# Parse and extract entities
user_queries_df['parsed_json'] = user_queries_df['json'].apply(safe_parse_json)
user_queries_df['entities'] = user_queries_df['parsed_json'].apply(extract_entities)

# Show distribution
all_entities_flat = [e for ents in user_queries_df['entities'] for e in ents]
entity_counts = Counter(all_entities_flat)
print("Entity Distribution:")
for entity, count in entity_counts.most_common():
    print(f"  {entity}: {count} ({100*count/len(user_queries_df):.1f}%)")

multi_entity = sum(1 for e in user_queries_df['entities'] if len(e) > 1)
print(f"\nQueries with multiple entities: {multi_entity} ({100*multi_entity/len(user_queries_df):.1f}%)")

In [ ]:
# Prepare train/test split
ALL_ENTITIES = sorted(list(set(all_entities_flat)))
print(f"All entity types ({len(ALL_ENTITIES)}): {ALL_ENTITIES}")

mlb = MultiLabelBinarizer(classes=ALL_ENTITIES)
y_encoded = mlb.fit_transform(user_queries_df['entities'])

# Split: 80% train, 20% test
X = user_queries_df['question'].tolist()
y = y_encoded
entities_list = user_queries_df['entities'].tolist()

indices = list(range(len(X)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_train = [X[i] for i in train_idx]
X_test = [X[i] for i in test_idx]
y_train = y[train_idx]
y_test = y[test_idx]
train_entities = [entities_list[i] for i in train_idx]
test_entities = [entities_list[i] for i in test_idx]

print(f"\nTrain size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

## 3. Evaluation Framework

In [ ]:
def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, 
                        label_names: List[str], method_name: str = "Method") -> Dict:
    """Compute and display evaluation metrics."""
    metrics = {
        'exact_match': accuracy_score(y_true, y_pred),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'jaccard_micro': jaccard_score(y_true, y_pred, average='micro', zero_division=0),
    }
    
    print(f"\n{'='*60}")
    print(f"RESULTS: {method_name}")
    print(f"{'='*60}")
    print(f"Exact Match Accuracy: {metrics['exact_match']:.4f}")
    print(f"F1 Micro:             {metrics['f1_micro']:.4f}")
    print(f"F1 Macro:             {metrics['f1_macro']:.4f}")
    print(f"Precision Micro:      {metrics['precision_micro']:.4f}")
    print(f"Recall Micro:         {metrics['recall_micro']:.4f}")
    print(f"Jaccard Micro:        {metrics['jaccard_micro']:.4f}")
    print(f"Hamming Loss:         {metrics['hamming_loss']:.4f}")
    print(f"\nPer-Class Report:")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    
    return metrics

---
## 4. Build RAG Retrieval System

This will be used by Approach 2 at inference time.

In [ ]:
class RAGRetriever:
    """
    Retrieves similar examples from training data using embeddings.
    """
    
    def __init__(self, embedding_model: str = "all-MiniLM-L6-v2"):
        print(f"Loading embedding model: {embedding_model}")
        self.embedding_model = SentenceTransformer(embedding_model)
        self.train_embeddings = None
        self.train_queries = None
        self.train_entities = None
    
    def index(self, queries: List[str], entities: List[List[str]]):
        """Index training queries for retrieval."""
        print(f"Indexing {len(queries)} training queries...")
        self.train_queries = queries
        self.train_entities = entities
        self.train_embeddings = self.embedding_model.encode(
            queries, show_progress_bar=True, convert_to_numpy=True
        )
        print("Indexing complete!")
    
    def retrieve(self, query: str, top_k: int = 5) -> List[Tuple[str, List[str], float]]:
        """Retrieve top-k most similar training examples."""
        query_embedding = self.embedding_model.encode([query], convert_to_numpy=True)[0]
        
        # Cosine similarity
        similarities = np.dot(self.train_embeddings, query_embedding) / (
            np.linalg.norm(self.train_embeddings, axis=1) * np.linalg.norm(query_embedding) + 1e-8
        )
        
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append((
                self.train_queries[idx],
                self.train_entities[idx],
                float(similarities[idx])
            ))
        return results
    
    def format_context(self, retrieved: List[Tuple[str, List[str], float]]) -> str:
        """Format retrieved examples as context string."""
        context_parts = ["\n[Similar examples]"]
        for query, entities, score in retrieved:
            context_parts.append(f"Query: {query} => Entities: {entities}")
        return "\n".join(context_parts)

In [ ]:
# Initialize and index the RAG retriever
rag_retriever = RAGRetriever()
rag_retriever.index(X_train, train_entities)

In [ ]:
# Test retrieval
test_query = "What SMS messages were sent from suspicious phones?"
retrieved = rag_retriever.retrieve(test_query, top_k=3)
print(f"Query: {test_query}")
print(f"\nRetrieved examples:")
for q, ents, score in retrieved:
    print(f"  [{score:.3f}] {q[:60]}... => {ents}")
print(f"\nFormatted context:")
print(rag_retriever.format_context(retrieved))

---
## 5. Fine-tuned Classifier

Train DistilBERT for multi-label classification on query text only.

In [ ]:
class EntityDataset(Dataset):
    """PyTorch Dataset for entity classification."""
    
    def __init__(self, texts: List[str], labels: np.ndarray, tokenizer, max_length: int = 256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [ ]:
class EntityClassifier:
    """
    Fine-tuned classifier for entity prediction.
    Supports both standard inference and RAG-augmented inference.
    """
    
    def __init__(self, model_name: str = "distilbert-base-uncased", num_labels: int = 9):
        self.model_name = model_name
        self.num_labels = num_labels
        self.device = DEVICE
        self.model = None
        self.tokenizer = None
        self.label_names = ALL_ENTITIES
    
    def train(self, X_train: List[str], y_train: np.ndarray, 
              epochs: int = 10, batch_size: int = 16, max_length: int = 128):
        """Train the classifier on query text only."""
        print(f"\n{'='*60}")
        print("TRAINING: Fine-tuned Classifier")
        print(f"{'='*60}")
        print(f"Model: {self.model_name}")
        print(f"Training samples: {len(X_train)}")
        print(f"Epochs: {epochs}")
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="multi_label_classification"
        )
        
        # Split for validation
        X_t, X_v, y_t, y_v = train_test_split(X_train, y_train, test_size=0.15, random_state=42)
        
        train_dataset = EntityDataset(X_t, y_t, self.tokenizer, max_length)
        val_dataset = EntityDataset(X_v, y_v, self.tokenizer, max_length)
        
        training_args = TrainingArguments(
            output_dir='./classifier_output',
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=100,
            weight_decay=0.01,
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_loss',
            report_to='none',
            fp16=torch.cuda.is_available(),
        )
        
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
        )
        
        print("\nTraining...")
        trainer.train()
        print("Training complete!")
        
        self.model.eval()
        self.model.to(self.device)
    
    def predict_single(self, text: str, threshold: float = 0.5, max_length: int = 256) -> Tuple[List[str], np.ndarray]:
        """Predict entities for a single text input."""
        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=max_length
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
        
        predicted = [self.label_names[i] for i, p in enumerate(probs) if p > threshold]
        return predicted if predicted else [self.label_names[np.argmax(probs)]], probs
    
    def predict_standard(self, queries: List[str], threshold: float = 0.5) -> List[List[str]]:
        """
        APPROACH 1: Standard prediction - query text only.
        """
        predictions = []
        for query in queries:
            pred, _ = self.predict_single(query, threshold)
            predictions.append(pred)
        return predictions
    
    def predict_with_rag(self, queries: List[str], retriever: RAGRetriever, 
                         top_k: int = 5, threshold: float = 0.5) -> List[List[str]]:
        """
        APPROACH 2: Prediction with RAG context at inference.
        Input format: query + retrieved similar examples
        """
        predictions = []
        for query in queries:
            # Retrieve similar examples
            retrieved = retriever.retrieve(query, top_k=top_k)
            context = retriever.format_context(retrieved)
            
            # Augmented input: query + RAG context
            augmented_input = f"{query}\n{context}"
            
            pred, _ = self.predict_single(augmented_input, threshold, max_length=512)
            predictions.append(pred)
        return predictions
    
    def save(self, path: str = './entity_classifier_model'):
        """Save the model."""
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"Model saved to {path}")
    
    def load(self, path: str = './entity_classifier_model'):
        """Load a saved model."""
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSequenceClassification.from_pretrained(path)
        self.model.eval()
        self.model.to(self.device)
        print(f"Model loaded from {path}")

In [ ]:
# Initialize and train the classifier
classifier = EntityClassifier(num_labels=len(ALL_ENTITIES))
classifier.train(X_train, y_train, epochs=10, batch_size=16)

In [ ]:
# Save the model
classifier.save('./entity_classifier_model')

---
## 6. Evaluate Both Approaches

In [ ]:
print("\n" + "="*70)
print("APPROACH 1: Standard Classifier (Query Only)")
print("="*70)
print("Running predictions on test set...")

# Approach 1: Standard prediction
predictions_standard = classifier.predict_standard(X_test)
y_pred_standard = mlb.transform(predictions_standard)

metrics_standard = evaluate_predictions(y_test, y_pred_standard, ALL_ENTITIES, 
                                        "Approach 1: Standard Classifier")

In [ ]:
print("\n" + "="*70)
print("APPROACH 2: Classifier + RAG Context (at inference)")
print("="*70)
print("Running predictions on test set with RAG context...")

# Approach 2: Prediction with RAG context
predictions_rag = classifier.predict_with_rag(X_test, rag_retriever, top_k=5)
y_pred_rag = mlb.transform(predictions_rag)

metrics_rag = evaluate_predictions(y_test, y_pred_rag, ALL_ENTITIES,
                                   "Approach 2: Classifier + RAG")

---
## 7. Results Comparison

In [ ]:
print("\n" + "="*70)
print("COMPARISON: Standard vs RAG-Augmented")
print("="*70)

comparison_df = pd.DataFrame({
    'Metric': ['Exact Match', 'F1 Micro', 'F1 Macro', 'Precision', 'Recall', 'Jaccard', 'Hamming Loss'],
    'Standard Classifier': [
        metrics_standard['exact_match'],
        metrics_standard['f1_micro'],
        metrics_standard['f1_macro'],
        metrics_standard['precision_micro'],
        metrics_standard['recall_micro'],
        metrics_standard['jaccard_micro'],
        metrics_standard['hamming_loss']
    ],
    'Classifier + RAG': [
        metrics_rag['exact_match'],
        metrics_rag['f1_micro'],
        metrics_rag['f1_macro'],
        metrics_rag['precision_micro'],
        metrics_rag['recall_micro'],
        metrics_rag['jaccard_micro'],
        metrics_rag['hamming_loss']
    ]
})

# Calculate difference
comparison_df['Difference'] = comparison_df['Classifier + RAG'] - comparison_df['Standard Classifier']
comparison_df['Better'] = comparison_df.apply(
    lambda row: 'RAG' if (row['Difference'] > 0 and row['Metric'] != 'Hamming Loss') or 
                        (row['Difference'] < 0 and row['Metric'] == 'Hamming Loss') 
                else ('Standard' if row['Difference'] != 0 else 'Tie'), axis=1
)

print(comparison_df.to_string(index=False))

print("\n" + "-"*70)
print("SUMMARY:")
rag_wins = sum(1 for b in comparison_df['Better'] if b == 'RAG')
std_wins = sum(1 for b in comparison_df['Better'] if b == 'Standard')
print(f"  RAG wins on {rag_wins} metrics")
print(f"  Standard wins on {std_wins} metrics")

---
## 8. Test Cases Comparison

In [ ]:
# Define test cases
test_cases = [
    ("What SMS messages were sent from suspicious phones to 0549876543 containing 'urgent'?", ["CDR", "Phone"]),
    ("Find all calls made using 3G technology", ["CDR"]),
    ("Show me all tweets from accounts with 500 friends mentioning Tesla", ["Web Activity", "Web Actor"]),
    ("Which phones have been marked as suspicious?", ["Phone"]),
    ("Find all individuals with occupation 'engineer' born before July 1985", ["Person"]),
    ("Show me investigations that are open or created in the last 3 months", ["Investigation"]),
    ("Find insights containing 'money laundering' from the past month", ["Insight"]),
    ("List visitors whose travel document was issued before January 2020", ["EVisa Request"]),
    ("Get reports created in the past 3 days", ["Report"]),
    ("Find Instagram profiles with 100 followers using Israel phone number", ["Web Actor"]),
    ("List emails sent to phones associated with target Sarah Johnson", ["CDR", "Phone"]),
]

print("\n" + "="*70)
print("TEST CASES COMPARISON")
print("="*70)

def check_prediction(pred, expected):
    """Check if prediction matches expected."""
    pred_set, exp_set = set(pred), set(expected)
    if pred_set == exp_set:
        return "✓ EXACT"
    elif pred_set & exp_set:
        return "~ PARTIAL"
    else:
        return "✗ WRONG"

std_correct = 0
rag_correct = 0

for query, expected in test_cases:
    # Standard prediction
    std_pred, _ = classifier.predict_single(query)
    
    # RAG prediction
    retrieved = rag_retriever.retrieve(query, top_k=5)
    context = rag_retriever.format_context(retrieved)
    augmented = f"{query}\n{context}"
    rag_pred, _ = classifier.predict_single(augmented, max_length=512)
    
    std_result = check_prediction(std_pred, expected)
    rag_result = check_prediction(rag_pred, expected)
    
    if "EXACT" in std_result:
        std_correct += 1
    if "EXACT" in rag_result:
        rag_correct += 1
    
    print(f"\nQuery: {query[:65]}{'...' if len(query) > 65 else ''}")
    print(f"Expected: {expected}")
    print(f"  Standard:  {std_result:12} {std_pred}")
    print(f"  With RAG:  {rag_result:12} {rag_pred}")

print(f"\n" + "="*70)
print(f"Test Cases Summary:")
print(f"  Standard: {std_correct}/{len(test_cases)} exact matches")
print(f"  With RAG: {rag_correct}/{len(test_cases)} exact matches")

---
## 9. Experiment: Different RAG Top-K Values

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT: Effect of Top-K on RAG Performance")
print("="*70)

k_values = [1, 3, 5, 10]
k_results = []

for k in k_values:
    print(f"\nTesting top_k = {k}...")
    preds = classifier.predict_with_rag(X_test, rag_retriever, top_k=k)
    y_pred = mlb.transform(preds)
    
    f1 = f1_score(y_test, y_pred, average='micro', zero_division=0)
    exact = accuracy_score(y_test, y_pred)
    
    k_results.append({'k': k, 'F1 Micro': f1, 'Exact Match': exact})
    print(f"  F1 Micro: {f1:.4f}, Exact Match: {exact:.4f}")

k_df = pd.DataFrame(k_results)
print("\nSummary:")
print(k_df.to_string(index=False))

best_k = k_df.loc[k_df['F1 Micro'].idxmax(), 'k']
print(f"\nBest top_k value: {best_k}")

---
## 10. Summary and Conclusions

### Approaches Implemented

**Approach 1: Fine-tuned Classifier (Standard)**
- Training: DistilBERT trained on query text → entity labels
- Inference: Query only → Model → Predicted entities
- Pros: Fast, simple, no retrieval overhead
- Cons: No access to similar examples at inference

**Approach 2: Fine-tuned Classifier + RAG**
- Training: Same as Approach 1 (query text only)
- Inference: Query + Retrieved similar examples → Same Model → Predicted entities
- Pros: Leverages similar examples as hints
- Cons: Requires embedding index, slightly slower

### Key Insight
The model was trained on query text only, but at inference in Approach 2, it receives 
augmented input with similar examples. The pre-trained language understanding can 
potentially leverage this additional context even without explicit training on it.

### Metrics Explanation
- **Exact Match**: % of queries where ALL predicted entities exactly match ground truth
- **F1 Score**: Harmonic mean of precision and recall
- **Precision**: Of predicted entities, how many were correct
- **Recall**: Of actual entities, how many were predicted
- **Jaccard Score**: Intersection over union of predicted and true labels
- **Hamming Loss**: Fraction of incorrectly predicted labels (lower is better)

### Open Issues & Future Improvements
1. **Train with RAG context**: Train the model on augmented inputs (query + examples)
2. **Better embeddings**: Use domain-specific embeddings for retrieval
3. **Threshold tuning**: Optimize per-entity thresholds
4. **Ensemble**: Combine predictions from both approaches
5. **Larger model**: Use BERT-base or RoBERTa for better performance

In [ ]:
# Save comparison results
comparison_df.to_csv('approach_comparison_results.csv', index=False)
print("Results saved to approach_comparison_results.csv")